In [8]:
import os
import glob
import cv2
import pytesseract
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter
from zipfile import ZipFile
from pathlib import Path
from IPython.display import Markdown, display
from typing import Optional

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

In [101]:
data_dir = Path("invoices_dataset/invoices")
images = os.listdir(data_dir)
sample_img = data_dir/images[0]

if not os.path.exists(data_dir):
    with ZipFile("invoices_dataset/invoices_dataset.zip", "r") as f:
        f.extractall("invoices")

if os.path.exists(data_dir):
    print(f"Invoices dataset exists and contains {len(os.listdir(data_dir))} images")

Invoices dataset exists and contains 371 images


In [102]:
class InvoiceImg:
    """Process random image"""

    def __init__(self):
        self.data_dir = Path("invoices_dataset/invoices")
        self.images = os.listdir(data_dir)

    def get_random_image_path(self):
        """Get a random image from the dataset"""
        random_img = images[np.random.randint(0, len(self.images))]
        return self.data_dir/random_img
    
    def pil_preprocess(self, img_path=None, plot_img=False, use_random_img=True):
        """Preprocess a random image or user image via `PIL` library"""
        if use_random_img:
            img_path = self.get_random_image_path()
            img = Image.open(img_path)
        else:
            if img_path is not None:
                img = Image.open(img_path)
            else:
                return f"Image path cannot be {type(img_path)}"
        img_trans= img.convert("L")
        enhancer = ImageEnhance.Contrast(img_trans)
        img_trans = enhancer.enhance(2)
        img_trans.filter(ImageFilter.SHARPEN)
        # img_trans.filter(ImageFilter.SMOOTH)
        resizing_scale = 1500/ img_trans.width
        new_img_size = (int(img_trans.width*resizing_scale), int(img_trans.height*resizing_scale)) #forces the image width to be 1k
        img_trans = img_trans.resize((new_img_size), Image.LANCZOS)
    
        if plot_img:
            fig = plt.figure(figsize=(10, 8))
            plt.imshow(img_trans, cmap="gray")
            plt.axis("off");
        
        return img_trans
    
    def cv2_preprocess(self, img_path=None, use_random_img=True, plot_img=False):
        if use_random_img:
            img_path = self.get_random_image_path()
            img=  cv2.imread(img_path)
        else:
            if img_path is not None:
                img = cv2.imread(img_path)
            else:
                return f"Image path cannot be {type(img_path)}"
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img = cv2.fastNlMeansDenoising(img)
        _, threshold = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        coords = np.column_stack(np.where(threshold>0))
        angle = cv2.minAreaRect(coords)[-1]
        if angle < -45:
            angle = (-90+angle)
        else:
            angle = -angle
        (h, w) = threshold.shape[:2]
        center = (w //2, h//2)
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        rotated = cv2.warpAffine(threshold, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

        if plot_img:
            cv2.imshow("Preprocessed img", rotated)
            cv2.waitKey(0)
            cv2.destroyAllWindows()
        return rotated
    
invoice = InvoiceImg()
invoice.get_random_image_path()

WindowsPath('invoices_dataset/invoices/X51005677333.jpg')

In [98]:
proccessed_img = invoice.cv2_preprocess(plot_img=True, use_random_img=True)

In [99]:
pytesseract.image_to_string(proccessed_img)

'184 SHVUILE Woe UV IOUUE PUeo\n\n005002 (BATANGKALI-2)\n\n002 28/03/2018 1\nOPEN CODE-SR ITEM\n\n0025679 U 3x8. 00\n0025679 U 147.00\n\nPD 3,20\n\nAuthorize : BATANGKALI-S\n\nSTAR 12X13 (1180)\n\n0020323 PKT 1x1. 00 1.00 §\nSTAR 15X16 AA (1X120)\n\n0020324 PRT 1x1,80\n\nTten 4 SubTotal Inc] G61\n\nQty 6 Spec. Disc\n\nSaving 3,80 Rounding\n\nTota)\n\n'

array([[255, 255, 255, ..., 255, 255, 255],
       [255, 255, 255, ..., 255, 255, 255],
       [255, 255, 255, ..., 255, 255, 255],
       ...,
       [255, 255, 255, ..., 255, 255, 255],
       [255, 255, 255, ..., 255, 255, 255],
       [255, 255, 255, ..., 255, 255, 255]],
      shape=(1023, 612), dtype=uint8)

In [31]:
def show_pil_img(img):
    _ = plt.figure(figsize=(14, 8))
    plt.imshow(img)
    
def show_cv2_img(img):
    cv2.imshow("Processed img", img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

In [35]:
show_cv2_img(rotated)